[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qualia906/Developing-Agentic-AI-with-LangChain/blob/main/chap05/solution/problem/chap05_solution.ipynb)

# 第5章 演習（正解）: マルチエージェントシステムの構築

---

In [ ]:
%pip install -qU langchain langchain-openai langgraph

In [ ]:
import os
from google.colab import userdata

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "chap05-solution"
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("環境変数の設定が完了しました")

## 課題1の正解: サブエージェントを3つ作成する

In [ ]:
# ✅ 正解
from langchain.agents import create_agent

research_agent = create_agent(
    model="openai:gpt-4o",
    tools=[],
    system_prompt=(
        "あなたはリサーチの専門家です。\n"
        "与えられたトピックについて、重要なポイントを3〜5つ箇条書きでまとめてください。\n"
        "事実に基づいた正確な情報を提供してください。"
    ),
)

writer_agent = create_agent(
    model="openai:gpt-4o",
    tools=[],
    system_prompt=(
        "あなたはテクノロジー分野のプロのライターです。\n"
        "提供されたリサーチ情報をもとに、一般のビジネスパーソン向けの\n"
        "わかりやすい記事（300〜500文字）を執筆してください。\n"
        "見出し、本文の構成を工夫して、読みやすい文章にしてください。"
    ),
)

validator_agent = create_agent(
    model="openai:gpt-4o",
    tools=[],
    system_prompt=(
        "あなたはコンテンツレビューの専門家です。\n"
        "提供された記事を以下の観点でレビューし、スコアとフィードバックを提供してください：\n"
        "1. 正確性（情報は正しいか）: /10\n"
        "2. 読みやすさ（文章は明瞭か）: /10\n"
        "3. 完成度（構成・内容は十分か）: /10\n"
        "最後に総合スコアと改善提案を述べてください。"
    ),
)

print("3つのサブエージェントの作成が完了しました")

## 課題2の正解: サブエージェントをツールとしてラップする

In [ ]:
# ✅ 正解
from langchain.tools import tool

@tool
def research(topic: str) -> str:
    """指定されたトピックについてリサーチを行い、重要なポイントをまとめます。
    
    Args:
        topic: リサーチするトピック
    """
    result = research_agent.invoke({
        "messages": [
            {"role": "user", "content": f"以下のトピックについてリサーチしてください：{topic}"}
        ]
    })
    return result["messages"][-1].content


@tool
def write_article(research_result: str, topic: str) -> str:
    """リサーチ結果をもとに、指定トピックの記事を執筆します。
    
    Args:
        research_result: リサーチ結果
        topic: 記事のトピック
    """
    result = writer_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": (
                    f"トピック: {topic}\n\n"
                    f"リサーチ結果:\n{research_result}\n\n"
                    "上記の情報をもとに記事を執筆してください。"
                )
            }
        ]
    })
    return result["messages"][-1].content


@tool
def validate_article(article: str) -> str:
    """執筆された記事の品質をチェックし、スコアとフィードバックを返します。
    
    Args:
        article: チェックする記事の本文
    """
    result = validator_agent.invoke({
        "messages": [
            {"role": "user", "content": f"以下の記事をレビューしてください：\n\n{article}"}
        ]
    })
    return result["messages"][-1].content


print("サブエージェントのツールラップが完了しました")

## 課題3の正解: Supervisor エージェントを作成して実行する

In [ ]:
# ✅ 正解
supervisor_agent = create_agent(
    model="openai:gpt-4o",
    tools=[research, write_article, validate_article],   # サブエージェントツールを渡す
    system_prompt=(
        "あなたはコンテンツ制作チームのスーパーバイザーです。\n"
        "ユーザーからトピックが与えられたら、以下のステップで記事制作を管理してください：\n"
        "1. research ツールでトピックをリサーチする\n"
        "2. write_article ツールでリサーチ結果をもとに記事を執筆する\n"
        "3. validate_article ツールで記事の品質をチェックする\n"
        "4. 最終的な記事と品質レポートをユーザーに提示する\n"
        "日本語で回答してください。"
    ),
)

result = supervisor_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "『LangChain を使った企業の AI 活用事例』というトピックで記事を作成してください"
        }
    ]
})

print(result["messages"][-1].content)

## 確認の正解: メッセージフロー

In [ ]:
# ✅ 正解
print(f"総メッセージ数: {len(result['messages'])}")
for i, msg in enumerate(result["messages"]):
    msg_type = type(msg).__name__
    print(f"[{i+1}] {msg_type}", end=" ")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        tool_names = [tc['name'] for tc in msg.tool_calls]
        print(f"→ ツール呼び出し: {tool_names}")
    elif msg_type == "ToolMessage":
        print(f"→ [{msg.name}] ({len(msg.content)}文字)")
    else:
        preview = msg.content[:60] + "..." if len(msg.content) > 60 else msg.content
        print(f"| {preview}")